# Section 4 — Object-Oriented Programming

**PCAP-31-03 official section** — 12 exam items, 34% (largest section) of the exam.

Source: [Python Institute PCAP-31-03 syllabus](https://pythoninstitute.org/pcap-exam-syllabus). Explanations below are grounded in the [official Python documentation](https://docs.python.org/3/) (linked per topic) — read the explanation, then run and tweak the code.

---


## 4.1 Understand the object-oriented approach

A **class** is a blueprint; an **object** (or *instance*) is a concrete thing built from that blueprint. A **property** (attribute) stores data on an object; a **method** is a function that belongs to a class and operates on its instances. **Encapsulation** means bundling data and the methods that operate on it together, and controlling access to the internals. **Inheritance** lets a **subclass** reuse and extend a **superclass**'s behavior — the superclass is more general, the subclass more specific.

Docs: [9. Classes](https://docs.python.org/3/tutorial/classes.html)


In [ ]:
class Animal:                 # superclass / base class
    def speak(self):
        return 'Some generic sound'

class Dog(Animal):            # subclass / derived class - inherits from Animal
    def speak(self):          # overrides the superclass's method
        return 'Woof!'

generic = Animal()
rex = Dog()
print(generic.speak())        # uses Animal's own method
print(rex.speak())            # uses Dog's overridden method
print(isinstance(rex, Animal))  # a Dog IS-A(n) Animal, thanks to inheritance


## 4.2 Employ class and object properties

A variable assigned *inside the class body* (not inside a method) is a **class variable** — shared by every instance. A variable assigned via `self.x = ...` (typically inside `__init__`) is an **instance variable** — unique per object. If an instance and its class both define an attribute of the same name, the *instance's* value wins on lookup (`obj.__dict__` shows only that instance's own attributes; `ClassName.__dict__` shows the class's).

**Classic bug:** a *mutable* class variable (like a list) is shared by all instances — mutating it from one instance affects every instance, because there's only one list object. Use an instance variable (created fresh in `__init__`) instead.

Any name starting with two leading underscores and at most one trailing underscore (like `__secret`) triggers **name mangling**: Python silently renames it to `_ClassName__secret` inside the class body. This doesn't make it truly private (you can still access `obj._ClassName__secret`), but it *does* prevent accidental name clashes with subclasses.

Docs: [9.6. Private Variables](https://docs.python.org/3/tutorial/classes.html#private-variables)


In [ ]:
class Dog:
    kind = 'canine'          # class variable: shared by ALL Dog instances

    def __init__(self, name):
        self.name = name      # instance variable: unique to each Dog

d1, d2 = Dog('Fido'), Dog('Rex')
print(d1.kind, d2.kind)        # both share the same class variable
print(d1.name, d2.name)        # each has its own instance variable

# The mutable-class-variable trap:
class BuggyDog:
    tricks = []                        # DANGER: one list shared by every instance
    def add_trick(self, trick):
        self.tricks.append(trick)

a, b = BuggyDog(), BuggyDog()
a.add_trick('sit')
print(b.tricks)                        # 'sit' leaked into b -- they share the SAME list!

# Name mangling
class Account:
    def __init__(self, balance):
        self.__balance = balance       # becomes _Account__balance internally
    def get_balance(self):
        return self.__balance

acc = Account(100)
print(acc.get_balance())
print(acc.__dict__)                    # shows the mangled name '_Account__balance'


## 4.3 Equip a class with methods

A method is just a function defined inside a class. Its first parameter is conventionally named `self` and refers to the specific instance the method was called on — Python passes it automatically: `obj.method(x)` is really `Class.method(obj, x)` behind the scenes. `self` isn't a keyword; the name is just a strong convention every Python programmer follows.


In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):                 # self refers to the instance calling area()
        return self.width * self.height

    def scale(self, factor):
        self.width *= factor
        self.height *= factor

r = Rectangle(3, 4)
print(r.area())            # instance method call: Python auto-passes r as self
print(Rectangle.area(r))   # exactly equivalent, written explicitly
r.scale(2)
print(r.area())


## 4.4 Discover the class structure

Python supports *introspection*: examining an object's structure at runtime. `hasattr(obj, 'name')` checks whether an attribute exists without raising an error. `type(obj).__name__` (or `obj.__class__.__name__`) gives the class name as a string. `SomeClass.__module__` shows which module defined the class. `SomeClass.__bases__` is a tuple of the class's immediate parent classes.


In [ ]:
class Vehicle:
    pass

class Car(Vehicle):
    def __init__(self):
        self.wheels = 4

my_car = Car()
print(hasattr(my_car, 'wheels'), hasattr(my_car, 'engine'))
print(type(my_car).__name__)         # 'Car'
print(Car.__module__)                 # where Car was defined
print(Car.__bases__)                  # (<class '__main__.Vehicle'>,)
print(Car.__bases__[0].__name__)


## 4.5 Build a class hierarchy using inheritance

Python supports both **single inheritance** (`class B(A)`) and **multiple inheritance** (`class C(A, B)`, more than one direct parent). `isinstance(obj, Cls)` checks whether `obj` belongs to `Cls` or any subclass; `is` / `is not` check *object identity* (same object in memory), which is a completely different thing from `==` (equal value). **Polymorphism** means different classes can be used interchangeably if they implement the same method names — calling `.speak()` on any `Animal` subclass "just works," even though each does something different. Overriding `__str__(self)` controls what `str(obj)` / `print(obj)` shows.

The **diamond problem**: if `B` and `C` both inherit from `A`, and `D` inherits from both `B` and `C`, which version of a method does `D` use if both `B` and `C` override it? Python resolves this deterministically using the **MRO** (Method Resolution Order, the C3 linearization algorithm) — visible via `D.__mro__` or `D.mro()`. Roughly: depth-first, left-to-right, without visiting the same class twice.

Docs: [9.5. Inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance)


In [ ]:
class A:
    def hello(self):
        return 'hello from A'

class B(A):
    def hello(self):
        return 'hello from B'

class C(A):
    def hello(self):
        return 'hello from C'

class D(B, C):   # multiple inheritance -- diamond shape: D -> B,C -> A
    pass

d = D()
print(d.hello())          # resolved via MRO -- B comes before C, so B wins
print(D.__mro__)          # the exact lookup order Python uses: D, B, C, A, object

print(isinstance(d, A), isinstance(d, C))   # True, True -- D IS-A B, C, and A
x = D()
y = d
print(x is d, y is d)      # is/is not check IDENTITY, not equality
print(x == d)              # different objects -> default __eq__ is also identity-based here

# Polymorphism + __str__ override
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __str__(self):
        return f'Point({self.x}, {self.y})'

p = Point(1, 2)
print(p)            # uses our __str__ instead of the default <__main__.Point object at ...>


## 4.6 Construct and initialize objects

`__init__(self, ...)` is the constructor: Python calls it automatically right after creating a new instance, to set up its initial state. It's not the same as `__new__` (which actually allocates the object — rarely overridden, out of scope here). A subclass's `__init__` can call `super().__init__(...)` to reuse the parent class's setup logic instead of duplicating it.


In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
        print(f'Animal.__init__ called for {name}')

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)     # reuse the parent constructor instead of repeating self.name = name
        self.breed = breed
        print(f'Dog.__init__ called for {breed}')

rex = Dog('Rex', 'Labrador')
print(rex.name, rex.breed)
